### GENOMICS - Integrating raw alleles into database

This pipeline cleans the allele data extracted manually from laboratory run records. 

#### 1. Write the original, raw Excel into an SQL database

Write the Excel to SQL for easier handling. At this step, the metadata, version and every single one of the 'RUN' sheets are saved as separate tables of a SQLite database. 

In [ ]:
import pandas as pd
import sqlite3
import os

# ==========================================
# CONFIGURATION
# ==========================================
# Use a raw string (r"...") so Windows backslashes don't cause errors

# Define excel file path. Update this path to point to your actual Excel file.
excel_file_path = r"C:\path\to\all_alleles_raw_lab_outputs_26Jun26.xlsx"

# Name your output database. This will be saved to the same folder as the Excel file. You can change this whenever you want.
sqlite_db_name = "alleles_raw_multisheet.sqlite"

# Automatically save the SQL database in the same folder as the Excel file
output_directory = os.path.dirname(excel_file_path)
sqlite_db_path = os.path.join(output_directory, sqlite_db_name)

# ==========================================
# EXECUTION
# ==========================================

# Step 1: Read all sheets from the Excel file
print(f"Reading Excel file: {excel_file_path}...")
print("This might take a minute or two because there are 250+ sheets to process.")

# sheet_name=None is the magic parameter that loads all sheets at once
all_sheets_dict = pd.read_excel(excel_file_path, sheet_name=None)
print(f"Successfully loaded {len(all_sheets_dict)} sheets into memory.")

# Step 2: Connect to (or create) the SQLite database
print(f"\nConnecting to SQLite database at: {sqlite_db_path}")
connection = sqlite3.connect(sqlite_db_path)

# Step 3: Loop through each sheet and save it as a table in SQL
print("Saving sheets to the database...")
for sheet_name, df in all_sheets_dict.items():
    
    # Write the dataframe to SQL. The table name will match the Excel sheet name.
    # if_exists="replace" ensures that if you run this twice, it overwrites rather than duplicating data.
    df.to_sql(name=sheet_name, con=connection, if_exists="replace", index=False)
    
    # Optional: Print out the progress so you can see it working
    print(f" - Saved table: {sheet_name} ({len(df)} rows)")

# Step 4: Close the database connection
connection.close()
print("\nAll done! The raw database is ready.")

Reading Excel file: C:\Users\miksa.henkrich\OneDrive - IMDEANU\Escritorio\PNK\dataprep\experiment_allele_integration_24Jun26\all_alleles_raw_lab_outputs_26Jun26.xlsx...
This might take a minute or two because there are 250+ sheets to process.
Successfully loaded 277 sheets into memory.

Connecting to SQLite database at: C:\Users\miksa.henkrich\OneDrive - IMDEANU\Escritorio\PNK\dataprep\experiment_allele_integration_24Jun26\alleles_raw_multisheet.sqlite
Saving sheets to the database...
 - Saved table: metadata (20 rows)
 - Saved table: version (9 rows)
 - Saved table: R256 (21 rows)
 - Saved table: R255 (21 rows)
 - Saved table: R254 (21 rows)
 - Saved table: R253 (21 rows)
 - Saved table: R252 (21 rows)
 - Saved table: R251 (21 rows)
 - Saved table: R250 (21 rows)
 - Saved table: R249 (21 rows)
 - Saved table: R248 (21 rows)
 - Saved table: R247 (21 rows)
 - Saved table: R246 (21 rows)
 - Saved table: R245 (21 rows)
 - Saved table: R244 (21 rows)
 - Saved table: R243 (21 rows)
 - Saved

#### 2. Test - melt a single sheet in the SQL from a wide to a long format

The next step is to transform the multi-sheet, wide format tables into a single, unified long table with 20 rows per sample for each SNP. First, test the table pivot from wide to long on a single sheet. This is just a check and can be skipped. 

In [ ]:
import pandas as pd
import sqlite3

# ==========================================
# CONFIGURATION
# ==========================================
# Define the path to your SQLite database. Update this path if your database is located elsewhere.
sqlite_raw_db = r"C:\path_to\alleles_raw_multisheet.sqlite"

# Define a sheet to test the melt process on. You can change this to any sheet name from your Excel file.
test_sheet_name = "R256"  # Change this if your first sheet has a different name

# ==========================================
# TEST MELT ON A SINGLE SHEET
# ==========================================
print(f"Testing melt process on sheet: {test_sheet_name}")

# 1. Load the single table from SQL
connection = sqlite3.connect(sqlite_raw_db)
query = f"SELECT * FROM '{test_sheet_name}'"
df_raw = pd.read_sql_query(query, connection)
connection.close()

# 2. Extract sample IDs from the first row (index 0)
# The first column is 'SNPID' (which is NULL in row 0), so we ignore it when getting sample IDs
sample_ids = df_raw.iloc[0].tolist()

# 3. Rename columns using the extracted sample IDs
# We explicitly keep the first column name as 'rs_id'
new_columns = ['rs_id'] + sample_ids[1:]
df_raw.columns = new_columns

# 4. Clean up the dataframe
# Drop the first row (which we just used for headers)
df_clean = df_raw.drop(index=0).copy()

# Drop any columns where the header is None/NaN (these are the empty numbered columns from Excel)
# In pandas, if a column name is None or NaN, it means there was no sample ID there.
df_clean = df_clean.loc[:, df_clean.columns.notna()]

# Ensure we drop any completely empty columns just in case
df_clean = df_clean.dropna(axis=1, how='all')

# 5. Melt from wide to long format
# 'id_vars' is the column that stays fixed (rs_id)
# 'var_name' is the new column for the headers (sample_id)
# 'value_name' is the new column for the data cells (raw_call)
df_long = pd.melt(
    df_clean,
    id_vars=['rs_id'],
    var_name='sample_id',
    value_name='raw_call'
)

# 6. Add the run identifier
df_long['run_id'] = test_sheet_name

# Clean up empty strings or completely null calls that might have slipped through
df_long = df_long.dropna(subset=['raw_call', 'rs_id'])
df_long = df_long[df_long['raw_call'].str.strip() != '']

# Reorder columns to match your preferred structure
df_long = df_long[['sample_id', 'run_id', 'rs_id', 'raw_call']]

# Show the results to verify it worked
print(f"Original wide shape: {df_raw.shape}")
print(f"New long shape: {df_long.shape}")
print("\nFirst 5 rows of the clean long table:")
print(df_long)


Testing melt process on sheet: R256
Original wide shape: (21, 20)
New long shape: (140, 4)

First 5 rows of the clean long table:
    sample_id run_id       rs_id raw_call
0    10086643   R256   rs7498665     a2a1
1    10086643   R256    rs696217       a2
2    10086643   R256   rs1801260     a2a1
3    10086643   R256   rs1467568       a1
4    10086643   R256  rs17300539     a2a1
..        ...    ...         ...      ...
135  10061527   R256    rs601338     a2a1
136  10061527   R256   rs1799883       a2
137  10061527   R256   rs1800896     a2a1
138  10061527   R256   rs1801282       a2
139  10061527   R256    rs174537       a1

[140 rows x 4 columns]


#### 3. Transform - melt all separate, wide SQL data sheets into a single long table

This step actually transforms the multiple data sheets into a single long table, and saves it to a new database together with metadata tables. 

In [ ]:
import pandas as pd
import sqlite3

# ==========================================
# CONFIGURATION
# ==========================================

# Define the input and output database paths. Update these paths if your databases are located elsewhere.
input_db_path = r"C:\path_to\alleles_raw_multisheet.sqlite"
output_db_path = r"C:\path_to\alleles_long.sqlite"


# Tables to skip during melting (they will be copied to the new DB exactly as-is)
tables_to_ignore = ['metadata', 'version']

# The name of the new combined table that will hold all the runs
combined_table_name = "alleles_long"

# ==========================================
# EXECUTION
# ==========================================
print(f"Connecting to databases...")
conn_in = sqlite3.connect(input_db_path)
conn_out = sqlite3.connect(output_db_path)

# Get a list of all tables in the input database
cursor = conn_in.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
all_tables = [row[0] for row in cursor.fetchall()]

# We will store each melted dataframe in this list, then combine them at the end
all_long_dfs = []

print("Processing tables...")
for table_name in all_tables:
    
    # Read the current table
    df_raw = pd.read_sql_query(f"SELECT * FROM '{table_name}'", conn_in)
    
    # 1. Handle ignored tables (like metadata)
    if table_name in tables_to_ignore:
        print(f" - Copying '{table_name}' as-is...")
        df_raw.to_sql(table_name, conn_out, if_exists="replace", index=False)
        continue
        
    # 2. Handle data tables (R1, R2, etc.)
    print(f" - Melting run sheet: '{table_name}'...")
    
    # Extract sample IDs from the first row
    sample_ids = df_raw.iloc[0].tolist()
    
    # Rename columns
    new_columns = ['rs_id'] + sample_ids[1:]
    df_raw.columns = new_columns
    
    # Clean up the dataframe (drop row 0 and completely empty columns)
    df_clean = df_raw.drop(index=0).copy()
    df_clean = df_clean.loc[:, df_clean.columns.notna()]
    df_clean = df_clean.dropna(axis=1, how='all')
    
    # Melt from wide to long
    df_long = pd.melt(
        df_clean,
        id_vars=['rs_id'],
        var_name='lab_sample_id',
        value_name='raw_call'
    )
    
    # Add run identifier
    df_long['run_id'] = table_name
    
    # Drop empty or completely null calls
    df_long = df_long.dropna(subset=['raw_call', 'rs_id'])
    df_long = df_long[df_long['raw_call'].astype(str).str.strip() != '']
    
    # Force lab_sample_id to be a string to avoid merging errors between numbers and text
    df_long['lab_sample_id'] = df_long['lab_sample_id'].astype(str)
    
    # Reorder columns
    df_long = df_long[['lab_sample_id', 'run_id', 'rs_id', 'raw_call']]
    
    # Append the clean sheet to our master list
    all_long_dfs.append(df_long)

# ==========================================
# COMBINE AND SAVE
# ==========================================
print("\nCombining all melted sheets into a single table...")
final_long_df = pd.concat(all_long_dfs, ignore_index=True)

print(f"Saving '{combined_table_name}' to output database ({len(final_long_df)} rows)...")
final_long_df.to_sql(combined_table_name, conn_out, if_exists="replace", index=False)

# Close the database connections
conn_in.close()
conn_out.close()

print("All done! Your new database is ready for the translation step.")

Connecting to databases...
Processing tables...
 - Copying 'metadata' as-is...
 - Copying 'version' as-is...
 - Melting run sheet: 'R256'...
 - Melting run sheet: 'R255'...
 - Melting run sheet: 'R254'...
 - Melting run sheet: 'R253'...
 - Melting run sheet: 'R252'...
 - Melting run sheet: 'R251'...
 - Melting run sheet: 'R250'...
 - Melting run sheet: 'R249'...
 - Melting run sheet: 'R248'...
 - Melting run sheet: 'R247'...
 - Melting run sheet: 'R246'...
 - Melting run sheet: 'R245'...
 - Melting run sheet: 'R244'...
 - Melting run sheet: 'R243'...
 - Melting run sheet: 'R242'...
 - Melting run sheet: 'R241'...
 - Melting run sheet: 'R240'...
 - Melting run sheet: 'R238'...
 - Melting run sheet: 'R239'...
 - Melting run sheet: 'R237'...
 - Melting run sheet: 'R236'...
 - Melting run sheet: 'R235'...
 - Melting run sheet: 'R234'...
 - Melting run sheet: 'R233'...
 - Melting run sheet: 'R232'...
 - Melting run sheet: 'R231'...
 - Melting run sheet: 'R230'...
 - Melting run sheet: 'R229

#### 4. Decode: based on the metadata table, decode a1/a2 type allele notations to alphabetic nucleobase codes (A/T/G/C)

This step decodes lab-level 'a1/a2' type genotype notations to canonical, alphabetical base notations based on the metadata table. 

In [ ]:
import pandas as pd
import sqlite3
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================

# Define the path to your SQLite database. Update this path if your database is located elsewhere.
db_path = r"C:\path_to\alleles_long.sqlite"

input_data_table = "alleles_long"
metadata_table = "metadata"
output_table_name = "alleles_long_annotated"

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading data from database...")
conn = sqlite3.connect(db_path)

# Load the long data table
df_long = pd.read_sql_query(f"SELECT * FROM {input_data_table}", conn)

# Load the metadata table
df_meta = pd.read_sql_query(f"SELECT * FROM {metadata_table}", conn)

# ==========================================
# 2. PREPARE THE METADATA LOOKUP
# ==========================================
print("Preparing metadata lookup...")
# We only need specific columns from the metadata for annotation
cols_to_keep = ['rs_id', 'gene', 'a1', 'a2', 'a2a1', 'mutation_pnk', 'risk_allele_pnk', 'minor_allele_pnk', 'mutation_gnomad', 'minor_allele_gnomad', 'gnomad_maf_eu', 'flag', 'discussion']

# Handle potential naming inconsistencies in the metadata Excel (e.g. trailing spaces)
df_meta.columns = df_meta.columns.str.strip()

# Subset the metadata and rename the encoding columns to match your requested structure
df_lookup = df_meta[cols_to_keep].copy()
df_lookup = df_lookup.rename(columns={
    'a1': 'a1_meaning',
    'a2': 'a2_meaning',
    'a2a1': 'a2a1_meaning'
})

# ==========================================
# 3. MERGE AND CLEAN RAW CALLS
# ==========================================
print("Merging data with metadata...")
# Merge the long table with our lookup table based on the rs_id
df_annotated = pd.merge(df_long, df_lookup, on='rs_id', how='left')

# Normalize the raw calls so that formatting differences don't break the translation
# Lowercase it, remove any colons (a1:a2 -> a1a2), and strip spaces
df_annotated['clean_call'] = df_annotated['raw_call'].astype(str).str.lower().str.replace(':', '').str.strip()

# Standardize heterozygous notation (make sure all variants become 'a2a1')
df_annotated.loc[df_annotated['clean_call'] == 'a1a2', 'clean_call'] = 'a2a1'

# ==========================================
# 4. TRANSLATE GENOTYPES & CALCULATE RISK
# ==========================================
print("Translating genotypes and calculating risk load...")

# Step 4a: Translate the raw call into the alphabetic genotype
def translate_genotype(row):
    call = row['clean_call']
    if call == 'a1':
        return row['a1_meaning']
    elif call == 'a2':
        return row['a2_meaning']
    elif call == 'a2a1':
        return row['a2a1_meaning']
    else:
        return None # In case of missing or unrecognized data (e.g., 'nd', 'nan')

df_annotated['genotype'] = df_annotated.apply(translate_genotype, axis=1)

# Step 4b: Calculate the risk load (0, 1, or 2 copies of the risk allele)
def calculate_risk_load(row):
    # If we don't have a valid genotype or risk allele, we can't calculate risk
    if pd.isna(row['genotype']) or pd.isna(row['risk_allele_pnk']):
        return np.nan
    
    # Simply count how many times the risk allele (e.g., 'G') appears in the genotype (e.g., 'AG')
    return row['genotype'].count(row['risk_allele_pnk'])

df_annotated['risk_load'] = df_annotated.apply(calculate_risk_load, axis=1)

# ==========================================
# 5. FORMAT AND SAVE OUTPUT
# ==========================================
print("Formatting final table...")
# Select and order the exact columns you requested
final_columns = [
    # IDs
    'lab_sample_id', 
    'run_id', 
    'rs_id', 
    # Metadata for each SNP
    'gene',
    'a1_meaning', 
    'a2_meaning', 
    'a2a1_meaning', 
    'minor_allele_pnk', 
    'risk_allele_pnk', 
    # Actual genotype call and risk load for the given sample
    'raw_call', 
    'genotype', 
    'risk_load'
]
df_final = df_annotated[final_columns]

print(f"Saving '{output_table_name}' to database ({len(df_final)} rows)...")
df_final.to_sql(output_table_name, conn, if_exists="replace", index=False)

conn.close()
print("Success! Genotypes translated and risk loads calculated.")

Loading data from database...
Preparing metadata lookup...
Merging data with metadata...
Translating genotypes and calculating risk load...
Formatting final table...
Saving 'alleles_long_annotated' to database (83340 rows)...
Success! Genotypes translated and risk loads calculated.


#### 5. QC#1: sanity checks if so far so good - what can i test in the long-annotaded database? 

This is a set of quality control checks before integrating the genetic data with the rest of the clinical database. 

##### QC#1a. Technical QC - was data wrangling from raw to pivoted and annotated allele tables OK? 

Are there as many  many run IDs in the transformed database as run sheets in the raw one, excluding metadata sheets? 

Are there any SNPs in any samples that are not annotated or not annotated correctly? 

Are there duplicate samples? If yes, the latest one is kept. 

Do the reference columns in the transformed database (rsID, meaning of a1, a2 and a2a1, the risk allele, etc) repeat every 20 rows? 

In [ ]:
import pandas as pd
import sqlite3
import re

# ==========================================
# CONFIGURATION
# ==========================================
# Define the paths to your SQLite databases. Update these paths if your databases are located elsewhere.
raw_multisheet_db = r"C:\path_to\alleles_raw_multisheet.sqlite" # To check expected runs
annotated_long_db = r"C:\path_to\alleles_long.sqlite"      # Where our working data lives

input_table = "alleles_long_annotated"              # Created in the previous step
output_table = "alleles_long_structural_qc_pass"       # Cleaned output table

# ==========================================
# LOAD DATA
# ==========================================
print("Loading data for Structural QC...\n")
conn_raw = sqlite3.connect(raw_multisheet_db)
conn_long = sqlite3.connect(annotated_long_db)

df = pd.read_sql_query(f"SELECT * FROM {input_table}", conn_long)

# ==========================================
# TEST 1: RUN COUNT & NAME VERIFICATION
# ==========================================
print("=== TEST 1: RUN ID VERIFICATION ===")
# Get the expected run names directly from the raw database's table list
cursor = conn_raw.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
all_raw_tables = set([row[0] for row in cursor.fetchall()])
expected_runs = all_raw_tables - {'metadata', 'version'} # Exclude non-run tables

# Get the actual run names that made it into our long table
actual_runs = set(df['run_id'].unique())

print(f"Expected runs from raw DB: {len(expected_runs)}")
print(f"Actual runs in long table: {len(actual_runs)}")

missing_runs = expected_runs - actual_runs
extra_runs = actual_runs - expected_runs

if missing_runs:
    print(f"⚠️ WARNING: The following runs are missing from the output: {missing_runs}")
if extra_runs:
    print(f"⚠️ WARNING: The following extra runs appeared unexpectedly: {extra_runs}")
if not missing_runs and not extra_runs:
    print("✅ Run IDs perfectly match between raw sheets and the annotated table.\n")


# ==========================================
# TEST 2: MISSINGNESS & INVALID CALLS
# ==========================================
print("=== TEST 2: INVALID RAW CALLS ===")
# Normalize the raw call just for the check (lowercase, remove colons)
# This handles 'a1:a2', 'A1', 'a1a2' etc., so we only flag true errors
temp_clean_call = df['raw_call'].astype(str).str.lower().str.replace(':', '').str.strip()

# Valid variants of the raw calls
valid_calls = {'a1', 'a2', 'a2a1', 'a1a2'}

# Find rows where the call is NOT in our valid list
invalid_mask = ~temp_clean_call.isin(valid_calls)
invalid_calls_df = df[invalid_mask]

if len(invalid_calls_df) > 0:
    print(f"⚠️ WARNING: Found {len(invalid_calls_df)} rows with invalid raw calls!")
    print("Examples of invalid calls found:")
    print(invalid_calls_df['raw_call'].value_counts().head())
else:
    print("✅ All raw calls conform to a1, a2, or a2a1 formats (no missingness).\n")


# ==========================================
# TEST 3: DUPLICATE SAMPLES
# ==========================================
print("=== TEST 3: DUPLICATE SAMPLES ===")
# Find how many unique runs each sample appears in
runs_per_sample = df.groupby('lab_sample_id')['run_id'].nunique()
duplicate_samples = runs_per_sample[runs_per_sample > 1].index.tolist()

if duplicate_samples:
    print(f"⚠️ WARNING: Found {len(duplicate_samples)} samples that were sequenced in multiple runs.")
    
    # Show a preview of the duplicates and their runs
    for sample in duplicate_samples[:50]: # Just print the first 50 so we don't flood the notebook
        runs = df[df['lab_sample_id'] == sample]['run_id'].unique()
        print(f"   - Sample {sample} found in runs: {', '.join(runs)}")
    if len(duplicate_samples) > 50:
        print(f"   ... and {len(duplicate_samples) - 50} more.")
        
    print("\nApplying deduplication rule: Keeping the later run...")
    
    # Helper function to extract numbers from run_id so 'R10' comes after 'R2'
    def extract_run_number(run_str):
        numbers = re.findall(r'\d+', str(run_str))
        return int(numbers[0]) if numbers else 0

    # Create a sorting key, sort the dataframe, and drop duplicates keeping the last one
    df['run_sort_key'] = df['run_id'].apply(extract_run_number)
    df = df.sort_values(by=['lab_sample_id', 'rs_id', 'run_sort_key'])
    
    # Drop duplicates based on sample and rs_id, keeping the 'last' (highest run number)
    df_dedup = df.drop_duplicates(subset=['lab_sample_id', 'rs_id'], keep='last').copy()
    
    # Remove our temporary sorting key
    df_dedup = df_dedup.drop(columns=['run_sort_key'])
    print(f"✅ Deduplication complete. Rows reduced from {len(df)} to {len(df_dedup)}.\n")
else:
    print("✅ No duplicate samples found across runs.\n")
    df_dedup = df.copy()


# ==========================================
# TEST 4: BLOCK STRUCTURE VERIFICATION
# ==========================================
print("=== TEST 4: 20-ROW BLOCK STRUCTURE ===")
# Count the number of rows (SNPs) per sample in our deduplicated table
snps_per_sample = df_dedup.groupby('lab_sample_id')['rs_id'].nunique()

samples_with_missing_snps = snps_per_sample[snps_per_sample != 20]

if len(samples_with_missing_snps) > 0:
    print(f"⚠️ WARNING: {len(samples_with_missing_snps)} samples do not have exactly 20 SNPs!")
    print(samples_with_missing_snps.head())
else:
    print("✅ Block structure perfect: Every sample has exactly 20 distinct rs_id rows.\n")

# Verify that the metadata columns are fully populated (no blocks are missing their lookup data)
# Note: 'gene' was not included in your earlier requested list, so we check the columns we generated:
meta_columns = ['rs_id', 'gene', 'a1_meaning', 'a2_meaning', 'a2a1_meaning', 'minor_allele_pnk', 'risk_allele_pnk']
missing_meta = df_dedup[meta_columns].isna().sum()

if missing_meta.sum() > 0:
    print("⚠️ WARNING: Some metadata annotations are missing (check the translation step):")
    print(missing_meta[missing_meta > 0])
else:
    print("✅ Metadata columns are perfectly populated across all rows.\n")


# ==========================================
# SAVE THE QC'D TABLE
# ==========================================
print(f"Saving structurally sound data to '{output_table}'...")
df_dedup.to_sql(output_table, conn_long, if_exists="replace", index=False)

conn_raw.close()
conn_long.close()
print("Success! Structural QC finished.")

Loading data for Structural QC...

=== TEST 1: RUN ID VERIFICATION ===
Expected runs from raw DB: 275
Actual runs in long table: 275
✅ Run IDs perfectly match between raw sheets and the annotated table.

=== TEST 2: INVALID RAW CALLS ===
✅ All raw calls conform to a1, a2, or a2a1 formats (no missingness).

=== TEST 3: DUPLICATE SAMPLES ===
⚠️ WARNING: Found 12 samples that were sequenced in multiple runs.
   - Sample 10030042 found in runs: R89_b 152938, R88_b 134142
   - Sample 10030370 found in runs: R89_b 152938, R88_b 134142
   - Sample 10030448 found in runs: R89_b 152938, R88_b 134142
   - Sample 10031032 found in runs: R89_b 152938, R88_b 134142
   - Sample 10031803 found in runs: R89_b 152938, R88_b 134142
   - Sample 10038260 found in runs: R89_b 152938, R88_b 134142
   - Sample 10039724 found in runs: R89_b 152938, R88_b 134142
   - Sample 10043448 found in runs: R89_b 152938, R88_b 134142
   - Sample 10043523 found in runs: R89_b 152938, R88_b 134142
   - Sample 10050927 fou

##### QC#1b. Batch effects - are any runs systematically different from the rest? 

Are there batch effects? Are there runs with outstandingly different allele distributions from the rest? 

Note: this is only tested for runs with over 10 samples, as low-sample runs naturally have greater variance. 

In [ ]:
import pandas as pd
import sqlite3
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
# Define the path to your SQLite database. Update this path if your database is located elsewhere.
db_path = r"C:\path_to\alleles_long.sqlite"      # Where our working data lives

input_table = "alleles_long_structural_qc_pass"

# Minimum samples in a run to be considered for outlier flagging
# (Small runs fluctuate naturally, so we don't want false alarms)
MIN_SAMPLES_FOR_FLAGGING = 10

# ==========================================
# LOAD DATA
# ==========================================
print("Loading data for Batch Effect QC...\n")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {input_table}", conn)
conn.close()

# ==========================================
# CALCULATE METRICS PER RUN
# ==========================================
# 1. How many unique samples are in each run?
run_sizes = df.groupby('run_id')['lab_sample_id'].nunique().rename('sample_count')

# 2. What is the Heterozygosity rate per run?
# We check if raw_call is 'a2a1'
df['is_het'] = (df['raw_call'] == 'a2a1').astype(int)
het_rates = df.groupby('run_id')['is_het'].mean().rename('het_rate')

# 3. What is the Risk Allele Frequency (RAF) per run?
# risk_load is 0, 1, or 2. RAF is the total risk alleles divided by total possible (2 * N)
df['risk_allele_count'] = df['risk_load']
raf_rates = (df.groupby('run_id')['risk_allele_count'].mean() / 2.0).rename('risk_allele_freq')

# Combine into a single run-level summary dataframe
run_stats = pd.concat([run_sizes, het_rates, raf_rates], axis=1).reset_index()

# ==========================================
# IDENTIFY BATCH EFFECTS (OUTLIERS)
# ==========================================
print(f"Analyzing {len(run_stats)} runs for batch effects...\n")

# Filter for runs large enough to have stable statistics
stable_runs = run_stats[run_stats['sample_count'] >= MIN_SAMPLES_FOR_FLAGGING].copy()
ignored_runs = run_stats[run_stats['sample_count'] < MIN_SAMPLES_FOR_FLAGGING]

print(f"Skipping outlier detection on {len(ignored_runs)} runs because they have fewer than {MIN_SAMPLES_FOR_FLAGGING} samples.")
print(f"Evaluating the remaining {len(stable_runs)} stable runs.\n")

# Calculate the global averages across the stable runs
global_het_mean = stable_runs['het_rate'].mean()
global_het_std = stable_runs['het_rate'].std()

global_raf_mean = stable_runs['risk_allele_freq'].mean()
global_raf_std = stable_runs['risk_allele_freq'].std()

print(f"Global Baseline (Stable Runs):")
print(f" - Average Heterozygosity: {global_het_mean:.1%} (±{global_het_std:.1%})")
print(f" - Average Risk Allele Freq: {global_raf_mean:.1%} (±{global_raf_std:.1%})\n")

# Flag outliers: Anything that deviates by more than 3 standard deviations
# (Using 3 SD is a standard, robust statistical threshold for extreme anomalies)
stable_runs['het_zscore'] = np.abs((stable_runs['het_rate'] - global_het_mean) / global_het_std)
stable_runs['raf_zscore'] = np.abs((stable_runs['risk_allele_freq'] - global_raf_mean) / global_raf_std)

outlier_het = stable_runs[stable_runs['het_zscore'] > 3.0]
outlier_raf = stable_runs[stable_runs['raf_zscore'] > 3.0]

# ==========================================
# REPORT ANOMALIES
# ==========================================
print("=== BATCH EFFECT RESULTS ===")

if not outlier_het.empty:
    print(f"⚠️ WARNING: Found {len(outlier_het)} runs with EXTREME Heterozygosity:")
    for _, row in outlier_het.iterrows():
        print(f"   - {row['run_id']} (N={row['sample_count']}): {row['het_rate']:.1%} "
              f"(Expected ~{global_het_mean:.1%})")
else:
    print("✅ No heterozygosity batch effects detected.")

print("")

if not outlier_raf.empty:
    print(f"⚠️ WARNING: Found {len(outlier_raf)} runs with EXTREME Risk Allele Frequencies:")
    for _, row in outlier_raf.iterrows():
        print(f"   - {row['run_id']} (N={row['sample_count']}): {row['risk_allele_freq']:.1%} "
              f"(Expected ~{global_raf_mean:.1%})")
else:
    print("✅ No allele frequency batch effects detected.")

# Print the top 3 and bottom 3 runs for visual inspection, just in case
print("\n=== VISUAL CHECK: Top & Bottom 3 Runs by Heterozygosity ===")
print("Lowest Heterozygosity:")
print(stable_runs.sort_values('het_rate').head(3)[['run_id', 'sample_count', 'het_rate']].to_string(index=False))
print("\nHighest Heterozygosity:")
print(stable_runs.sort_values('het_rate').tail(3)[['run_id', 'sample_count', 'het_rate']].to_string(index=False))

Loading data for Batch Effect QC...

Analyzing 275 runs for batch effects...

Skipping outlier detection on 31 runs because they have fewer than 10 samples.
Evaluating the remaining 244 stable runs.

Global Baseline (Stable Runs):
 - Average Heterozygosity: 35.1% (±3.2%)
 - Average Risk Allele Freq: 38.3% (±2.2%)

=== BATCH EFFECT RESULTS ===
⚠️ WARNING: Found 3 runs with EXTREME Heterozygosity:
   - R3 (N=15): 45.0% (Expected ~35.1%)
   - R39L (N=18): 45.3% (Expected ~35.1%)
   - R4 (N=16): 52.2% (Expected ~35.1%)

⚠️ WARNING: Found 4 runs with EXTREME Risk Allele Frequencies:
   - R3 (N=15): 46.2% (Expected ~38.3%)
   - R38-2L (N=16): 47.3% (Expected ~38.3%)
   - R39L (N=18): 48.5% (Expected ~38.3%)
   - R4 (N=16): 45.8% (Expected ~38.3%)

=== VISUAL CHECK: Top & Bottom 3 Runs by Heterozygosity ===
Lowest Heterozygosity:
run_id  sample_count  het_rate
  R161            10  0.275000
 R87_b            16  0.278125
  R255            12  0.279167

Highest Heterozygosity:
run_id  sample_c

##### QC#1c. HWE+MAF - incorporating gnomAD MAF references too

Are the loci in Hardy-Weinberg equilibrium, ie. do the genotypes conform to the expected frequencies given the population size? 

Are the minor allele frequencies roughly similar to that of a non-Finnish European population according to an external reference (gnomAD)? A difference under 10% is tolerated. 

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
from scipy.stats import chisquare

# ==========================================
# CONFIGURATION
# ==========================================
# Define the path to your SQLite database. Update this path if your database is located elsewhere.
db_path = r"C:\path_to\alleles_long.sqlite"

input_table = "alleles_long_structural_qc_pass"
metadata_table = "metadata" # ADDED: Your metadata table

# The runs we mathematically identified in the previous step as failing chemistry/protocol
runs_to_drop = ['R3', 'R4', 'R38-2L', 'R39L']

# ADDED: Thresholds for MAF flagging
TOLERANCE_WARNING = 0.10
TOLERANCE_SEVERE = 0.35

# ==========================================
# 1. LOAD & FILTER DATA
# ==========================================
print("Loading data for Biological QC...\n")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {input_table}", conn)
df_meta = pd.read_sql_query(f"SELECT * FROM {metadata_table}", conn) # ADDED: Load metadata
conn.close()

# Drop the bad runs before doing biological statistics
initial_rows = len(df)
df_clean = df[~df['run_id'].isin(runs_to_drop)].copy()
dropped_rows = initial_rows - len(df_clean)

print(f"Removed {dropped_rows} rows from {len(runs_to_drop)} failed runs.")
print(f"Proceeding with {len(df_clean)} high-quality SNP calls.\n")

# ==========================================
# 2. CALCULATE MAF & HWE PER SNP
# ==========================================
print("Calculating Hardy-Weinberg Equilibrium & Allele Frequencies...\n")

qc_results = []

# Loop through each of the 20 SNPs one by one
unique_snps = df_clean['rs_id'].unique()

for rs_id in unique_snps:
    # Isolate data for just this SNP
    snp_data = df_clean[df_clean['rs_id'] == rs_id]
    
    # ------------------------------------------
    # A. Observed Genotype Counts
    # ------------------------------------------
    n_total = len(snp_data)
    
    # Count how many people have 0, 1, or 2 risk alleles
    obs_0 = len(snp_data[snp_data['risk_load'] == 0])
    obs_1 = len(snp_data[snp_data['risk_load'] == 1])
    obs_2 = len(snp_data[snp_data['risk_load'] == 2])
    
    # ------------------------------------------
    # B. Allele Frequencies
    # ------------------------------------------
    # Total risk alleles observed / Total possible alleles (2 per person)
    risk_allele_freq = (obs_1 + (2 * obs_2)) / (2 * n_total)
    non_risk_freq = 1.0 - risk_allele_freq
    
    # Minor Allele Frequency (MAF) is simply whichever frequency is smaller
    maf = min(risk_allele_freq, non_risk_freq)
    
    # ------------------------------------------
    # C. Expected Genotypes (Hardy-Weinberg Math)
    # ------------------------------------------
    # HW Equation: p^2 + 2pq + q^2 = 1
    exp_0 = (non_risk_freq ** 2) * n_total
    exp_1 = (2 * risk_allele_freq * non_risk_freq) * n_total
    exp_2 = (risk_allele_freq ** 2) * n_total
    
    # ------------------------------------------
    # D. Statistical Test (Chi-Square)
    # ------------------------------------------
    obs_counts = [obs_0, obs_1, obs_2]
    exp_counts = [exp_0, exp_1, exp_2]
    
    # Run the test. A low p-value (< 0.000001) means observed deviates massively from expected
    chi2_stat, p_value = chisquare(f_obs=obs_counts, f_exp=exp_counts)
    
    # ------------------------------------------
    # E. ADDED: MAF Reference Check
    # ------------------------------------------
    meta_row = df_meta[df_meta['rs_id'] == rs_id].iloc[0]
    pnk_minor = meta_row['minor_allele_pnk']
    gnomad_minor = meta_row['minor_allele_gnomad']
    
    # Handle cases like "?" in the gnomad_maf_eu column
    try:
        gnomad_maf = float(meta_row['gnomad_maf_eu'])
    except ValueError:
        gnomad_maf = np.nan

    # Calculate actual observed frequency of the PNK minor allele
    total_pnk_minor_alleles = snp_data['genotype'].astype(str).str.count(pnk_minor).sum()
    obs_pnk_freq = total_pnk_minor_alleles / (2.0 * n_total)
    
    # Flag generation
    flag = "✅ OK"
    flag_reason = ""
    abs_diff = np.nan
    
    if pnk_minor != gnomad_minor:
        flag_reason = f"Mismatch (PNK:{pnk_minor} vs gnomAD:{gnomad_minor}). "
        
    if pd.notna(gnomad_maf):
        abs_diff = abs(obs_pnk_freq - gnomad_maf)
        if abs_diff >= TOLERANCE_SEVERE:
            if abs(obs_pnk_freq - (1.0 - gnomad_maf)) < TOLERANCE_WARNING:
                flag = "🛑 SEVERE (FLIPPED)"
                flag_reason += "PNK minor allele operates as MAJOR allele."
            else:
                flag = "🛑 SEVERE (DEVIATION)"
                flag_reason += "Massive deviation from reference."
        elif abs_diff >= TOLERANCE_WARNING:
            flag = "⚠️ WARNING"
            flag_reason += f"Deviation of {abs_diff:.1%} exceeds normal cohort enrichment."
        elif flag_reason != "":
            flag = "ℹ️ NOTE"
            flag_reason += "Frequencies match despite letter mismatch (Strand flip OK)."
    else:
        flag = "❓ UNKNOWN"
        flag_reason += "No gnomAD MAF provided."

    # Save the metrics to our list
    qc_results.append({
        'rs_id': rs_id,
        'n_samples': n_total,
        'risk_allele_freq': risk_allele_freq,
        'minor_allele_freq': maf,
        'observed_freq_0_1_2': f"{obs_0} | {obs_1} | {obs_2}",
        'expected_freq_0_1_2': f"{int(exp_0)} | {int(exp_1)} | {int(exp_2)}",
        'hwe_p_value': p_value,
        # ADDED: New output metrics
        'pnk_minor': pnk_minor,
        'gnomad_minor': gnomad_minor,
        'obs_pnk_freq': obs_pnk_freq,
        'gnomad_maf': gnomad_maf,
        'abs_diff': abs_diff,
        'flag': flag,
        'flag_details': flag_reason
    })

# Convert results to a clean dataframe
df_qc = pd.DataFrame(qc_results)

# ==========================================
# 3. REPORT RESULTS
# ==========================================
# Flag SNPs that severely violate HWE (Using standard GWAS threshold of 10^-6)
hwe_failures = df_qc[df_qc['hwe_p_value'] < 1e-6]

print("=== HWE TEST OUTCOMES ===")
print(df_qc[['rs_id', 'n_samples', 'expected_freq_0_1_2', 'observed_freq_0_1_2', 'hwe_p_value']].head(20).to_string(index=False))

if not hwe_failures.empty:
    print(f"\n⚠️ WARNING: {len(hwe_failures)} SNPs severely violate Hardy-Weinberg Equilibrium.")
    for _, row in hwe_failures.iterrows():
        print(f" - {row['rs_id']}: p-value = {row['hwe_p_value']:.2e}")
        print(f"      Observed (0|1|2): {row['observed_freq_0_1_2']}")
        print(f"      Expected (0|1|2): {row['expected_freq_0_1_2']}")
else:
    print("\n✅ All SNPs conform to Hardy-Weinberg expectations.")

# ADDED/CHANGED: Replaced the old preview with the new comprehensive reference check
print("\n=== MAF REFERENCE QC REPORT ===")
report_cols = ['rs_id', 'pnk_minor', 'gnomad_minor', 'obs_pnk_freq', 'gnomad_maf', 'flag', 'flag_details']
# Sort so severe issues bubble to the top
df_qc_sorted = df_qc.sort_values(by=['abs_diff'], ascending=False)
print(df_qc_sorted[report_cols].to_string(index=False))

print("\n(To inspect the full df_qc for all 20 SNPs, save or print it).")

Loading data for Biological QC...

Removed 1300 rows from 4 failed runs.
Proceeding with 81600 high-quality SNP calls.

Calculating Hardy-Weinberg Equilibrium & Allele Frequencies...

=== HWE TEST OUTCOMES ===
     rs_id  n_samples expected_freq_0_1_2 observed_freq_0_1_2  hwe_p_value
 rs1042713       4080   611 | 1936 | 1532   625 | 1909 | 1546     0.671531
 rs1421085       4080   1194 | 2026 | 859   1211 | 1993 | 876     0.577398
 rs1467568       4080   1744 | 1846 | 488   1754 | 1828 | 498     0.811707
rs17300539       4080     50 | 806 | 3223     54 | 799 | 3227     0.850403
  rs174537       4080   364 | 1709 | 2005   363 | 1713 | 2004     0.993699
rs17782313       4080   2499 | 1387 | 192   2490 | 1407 | 183     0.675728
 rs1799883       4080   2099 | 1654 | 326   2092 | 1669 | 319     0.859853
 rs1800206       4080     3417 | 633 | 29     3427 | 614 | 39     0.149925
 rs1800795       4080   1740 | 1848 | 490   1762 | 1806 | 512     0.339859
 rs1800896       4080   646 | 1955 | 147